In [ ]:
# ================================================================
# TASK 4 - OPTIMIZATION MODEL
# CODTECH IT Solutions - Data Science Internship
# Description : Product Mix Optimization using Linear Programming
# Library     : PuLP
# ================================================================

import pulp
import matplotlib.pyplot as plt
import numpy as np

print("PuLP version:", pulp.__version__)
print("✅ Libraries imported successfully!")

In [ ]:
# ================================================================
# BUSINESS PROBLEM
# A furniture factory produces Chairs and Tables.
# Goal: Maximize profit given limited wood and labor resources.
#
# Profit per unit:
#   Chair : $50
#   Table : $120
#
# Resource requirements per unit:
#            Wood(kg)  Labor(hrs)
#   Chair  :    2         3
#   Table  :    5         4
#
# Available resources:
#   Wood   : 200 kg
#   Labor  : 180 hours
# ================================================================

# Define the problem - we want to MAXIMIZE profit
prob = pulp.LpProblem("Product_Mix_Optimization", pulp.LpMaximize)

# Define decision variables
# These are the quantities we want to find
# lowBound=0 means we can't produce negative quantities
chairs = pulp.LpVariable("Chairs", lowBound=0, cat='Integer')
tables = pulp.LpVariable("Tables", lowBound=0, cat='Integer')

print("✅ Problem and variables defined!")
print(f"Decision variables: {chairs}, {tables}")

In [ ]:
# Objective function: Maximize total profit
# Profit = $50 per chair + $120 per table
prob += 50 * chairs + 120 * tables, "Total Profit"

# Constraints - resource limitations
# Wood constraint: each chair needs 2kg, each table needs 5kg
prob += 2 * chairs + 5 * tables <= 200, "Wood Constraint"

# Labor constraint: each chair needs 3hrs, each table needs 4hrs
prob += 3 * chairs + 4 * tables <= 180, "Labor Constraint"

print("✅ Objective function and constraints set!")
print("\nProblem Summary:")
print(prob)

In [ ]:
# Solve the optimization problem
prob.solve(pulp.PULP_CBC_CMD(msg=0))

# Check if solution was found
print(f"Status: {pulp.LpStatus[prob.status]}")

# Extract results
chairs_qty = int(chairs.value())
tables_qty = int(tables.value())
total_profit = 50 * chairs_qty + 120 * tables_qty

# Resource usage
wood_used  = 2 * chairs_qty + 5 * tables_qty
labor_used = 3 * chairs_qty + 4 * tables_qty

print("\n" + "=" * 45)
print("        OPTIMIZATION RESULTS")
print("=" * 45)
print(f"  Chairs to produce : {chairs_qty} units")
print(f"  Tables to produce : {tables_qty} units")
print(f"  Maximum Profit    : ${total_profit}")
print("=" * 45)
print(f"  Wood used         : {wood_used} / 200 kg")
print(f"  Labor used        : {labor_used} / 180 hrs")
print("=" * 45)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Production quantities
products = ['Chairs', 'Tables']
quantities = [chairs_qty, tables_qty]
colors = ['#3498db', '#2ecc71']

bars = axes[0].bar(products, quantities, color=colors,
                   edgecolor='black', width=0.4)
axes[0].set_title('Optimal Production Quantities', fontsize=13)
axes[0].set_ylabel('Units to Produce')
axes[0].set_ylim(0, max(quantities) + 10)
for bar, qty in zip(bars, quantities):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5,
                 str(qty), ha='center', fontweight='bold')

# Plot 2: Resource utilization
resources     = ['Wood (kg)', 'Labor (hrs)']
used          = [wood_used, labor_used]
available     = [200, 180]
remaining     = [200 - wood_used, 180 - labor_used]

x = np.arange(len(resources))
width = 0.35

axes[1].bar(x, used,      width, label='Used',      color='#e74c3c', edgecolor='black')
axes[1].bar(x, remaining, width, label='Remaining',  color='#95a5a6', edgecolor='black',
            bottom=used)
axes[1].set_title('Resource Utilization', fontsize=13)
axes[1].set_ylabel('Amount')
axes[1].set_xticks(x)
axes[1].set_xticklabels(resources)
axes[1].legend()

for i, (u, a) in enumerate(zip(used, available)):
    axes[1].text(i, a + 1, f'{u}/{a}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('optimization_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Visualization saved as optimization_results.png")

In [ ]:
# ================================================================
# BUSINESS INSIGHTS
# Translate the mathematical solution into actionable
# business recommendations for decision makers
# ================================================================

print("=" * 50)
print("         BUSINESS INSIGHTS & RECOMMENDATIONS")
print("=" * 50)

print(f"""
📦 OPTIMAL PRODUCTION PLAN:
   • Produce {chairs_qty} Chairs and {tables_qty} Tables
   • This yields the maximum profit of ${total_profit}

🪵 RESOURCE ANALYSIS:
   • Wood   : Fully utilized ({wood_used}/200 kg) — BOTTLENECK
   • Labor  : {labor_used}/180 hrs used — {180-labor_used} hrs remaining

💡 RECOMMENDATIONS:
   • Wood is the limiting resource — acquiring more wood
     would directly increase profit
   • {180-labor_used} labor hours are idle — workers could be
     used for other tasks or shifts reduced
   • Tables generate higher profit ($120 vs $50)
     so they are prioritized in the optimal mix

📈 PROFIT BREAKDOWN:
   • Chairs : {chairs_qty} × $50  = ${chairs_qty * 50}
   • Tables : {tables_qty} × $120 = ${tables_qty * 120}
   • Total  : ${total_profit}
""")
print("=" * 50)
print("✅ Optimization completed successfully!")
print("=" * 50)